# Digital channel — StegaStamp recovery and faster resume

Same frozen 29 methods, messages and 79 conditions. This release fixes four known empty StegaStamp PNGs only when their original hashes and Clean bits reproduce exactly. No reference hash or measured result is changed. Stop any old active runner first.

Upload the matching new source ZIP once; it stays on Drive. Use the same bank/output folders and GPU type. Run all cells. Existing complete methods are reused; the first run creates compact completion receipts, later runs avoid large catalogue reads.


In [ ]:
from pathlib import Path
import os, sys, json, hashlib, shutil, subprocess, zipfile

N_IMAGES = 100                  # After review: change ONLY this to 100; keep the same output.
MODELS = "all"                # All 29; or a list of exact method IDs for diagnostics.
MAX_HOURS = 24                # A session budget, NOT a guarantee of Colab uptime.
SSD_RESERVE_GIB = 60          # Leave SSD room for one isolated decoder environment.
PAIRWISE_TABLES = True
SOURCE_ARCHIVE = Path("COCO_Digital_Channel_RECOVERY_FAST_RESUME_source.zip")
DECODE_WORKERS = 4
ATTACK_WORKERS = 4
FULL_AUDIT = False

try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
if os.name != "posix":
    raise RuntimeError("Full 29-method execution requires Linux/NVIDIA: use Colab or a GPU server.")
if IN_COLAB:
    drive.mount("/content/drive")
BASE = Path("/content/drive/MyDrive") if IN_COLAB else Path("/workspace/data")
BANK_ROOT = BASE / "watermarking_production_bank_v1"       # READ ONLY: completed embedding bank.
OUTPUT_ROOT = BASE / "watermarking_digital_channel_v1"    # Durable measurements and reports.
SCRATCH_ROOT = Path("/content/wmbench_digital_scratch") if IN_COLAB else Path("/workspace/wmbench_digital_scratch")
SETTINGS = dict(n_images=N_IMAGES, models=MODELS, max_hours=MAX_HOURS, device="cuda",
    bank=str(BANK_ROOT), output=str(OUTPUT_ROOT), scratch=str(SCRATCH_ROOT),
    ssd_reserve_gib=SSD_RESERVE_GIB, pairwise=PAIRWISE_TABLES, download_author_assets=True,
    manual_assets_root=str(BANK_ROOT / "_manual_assets"))
print("Bank (read-only):", BANK_ROOT)
print("Results:", OUTPUT_ROOT)
print("Temporary fallback:", OUTPUT_ROOT / "_temporary")
print("Per method: save 78 attacked PNGs per available carrier in ZIPs → decode → commit → remove ZIPs.")

SETTINGS.update(decode_workers=DECODE_WORKERS, attack_workers=ATTACK_WORKERS, full_audit=FULL_AUDIT)


## Install

One small source ZIP, uploaded once and saved on Drive. No Base64. Frozen model environments remain isolated.


In [ ]:
# Upload the matching small source ZIP once; a copy is kept on Drive.
EXPECTED_SHA = "af9de00ac0fd35318d082cc5e0de3ac50dca4abc4a00a9d808aed158b30b0bd5"
SAVED_ZIP = OUTPUT_ROOT / "_release" / SOURCE_ARCHIVE.name
if not SOURCE_ARCHIVE.is_file() and SAVED_ZIP.is_file():
    SOURCE_ARCHIVE = SAVED_ZIP
if not SOURCE_ARCHIVE.is_file():
    if not IN_COLAB:
        raise FileNotFoundError("Place the matching source ZIP beside this notebook or set SOURCE_ARCHIVE.")
    from google.colab import files
    uploaded = files.upload()
    matches = [Path(name) for name in uploaded if hashlib.sha256(uploaded[name]).hexdigest() == EXPECTED_SHA]
    if len(matches) != 1:
        raise ValueError("Upload exactly the source ZIP supplied with this notebook.")
    SOURCE_ARCHIVE = matches[0]
data = SOURCE_ARCHIVE.read_bytes()
if hashlib.sha256(data).hexdigest() != EXPECTED_SHA:
    raise ValueError("Source ZIP checksum mismatch. Do not use a different notebook's ZIP.")
SAVED_ZIP.parent.mkdir(parents=True, exist_ok=True)
if not SAVED_ZIP.exists():
    SAVED_ZIP.write_bytes(data)
if hashlib.sha256(SAVED_ZIP.read_bytes()).hexdigest() != EXPECTED_SHA:
    raise ValueError("Saved release readback failed")
PROJECT = SCRATCH_ROOT / "performance_release_recovery_v2"
PROJECT.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(SOURCE_ARCHIVE) as archive:
    for member in archive.infolist():
        target = (PROJECT / member.filename).resolve()
        if not target.is_relative_to(PROJECT.resolve()) or member.is_dir():
            raise ValueError("Unsafe or unexpected source ZIP entry")
        content = archive.read(member)
        if target.exists() and target.read_bytes() != content:
            raise ValueError(f"Source changed: {target}. Do not overwrite it; keep bank/output paths and inspect the source conflict.")
        if not target.exists():
            target.parent.mkdir(parents=True, exist_ok=True)
            target.write_bytes(content)

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "uv==0.8.15"], check=True)
import uv
UV = str(uv.find_uv_bin())
controller_dir = SCRATCH_ROOT / "controller"
CONTROLLER = controller_dir / "bin/python"
if not CONTROLLER.exists():
    subprocess.run([UV, "venv", "--python", "3.10.12", str(controller_dir)], check=True)
# This is the CPU attack/report + quality-metric environment, NOT a model environment.
subprocess.run([UV, "pip", "install", "--python", str(CONTROLLER),
    "numpy==1.26.4", "Pillow==10.4.0", "scipy==1.11.4", "scikit-image==0.22.0",
    "psutil==6.1.1", "gdown==5.2.0", "torch==2.5.1", "torchvision==0.20.1", "lpips==0.1.4"], check=True)
ENV = os.environ.copy()
ENV.update(PYTHONPATH=os.pathsep.join([str(PROJECT / "src"), str(PROJECT / "production/src")]),
    PYTHONUNBUFFERED="1", OMP_NUM_THREADS="4", OPENBLAS_NUM_THREADS="1", MKL_NUM_THREADS="4",
    PATH=str(Path(UV).parent)+os.pathsep+ENV.get("PATH", ""),
    UV_PYTHON_INSTALL_DIR=str(SCRATCH_ROOT / "python"))
CONFIG = SCRATCH_ROOT / "digital-settings.json"
CONFIG.write_text(json.dumps(SETTINGS, indent=2), encoding="utf-8")
print("Free SSD: %.1f GiB" % (shutil.disk_usage(SCRATCH_ROOT).free / 1024**3))
subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"], check=True)


In [ ]:
# Only RoSteALS/SepWater need these author-import system libraries.
# Restore the setup used by RoSteALS_AUTHOR_FIX and RoSteALS_SepWater_RECOVERY.
# No model source, checkpoint, Python lock, payload or matching rule is changed.
import ctypes
import ctypes.util

def ensure_author_system_dependencies(models, in_colab):
    affected = {"rosteals", "sepwater"}
    selected = affected if models == "all" else affected.intersection(models)
    if not selected:
        return {}

    requirements = {
        "libmagickwand-dev": (
            ("MagickWand-6.Q16", "MagickWand-6.Q16HDRI", "MagickWand-7.Q16HDRI",
             "MagickWand-7.Q16", "MagickWand"), "MagickWandGenesis"),
        "libgl1": (("GL",), None),
    }

    def inspect_libraries():
        found = {}
        for package, (names, symbol) in requirements.items():
            for name in names:
                path = ctypes.util.find_library(name)
                if not path:
                    continue
                try:
                    library = ctypes.CDLL(path)
                    if symbol:
                        getattr(library, symbol)
                except (OSError, AttributeError):
                    continue
                found[package] = path
                break
        return found

    found = inspect_libraries()
    missing = [package for package in requirements if package not in found]
    if missing:
        install = ["apt-get", "-qq", "install", "-y", "--no-install-recommends", *missing]
        if not in_colab or not shutil.which("apt-get") or os.geteuid() != 0:
            raise RuntimeError(
                "RoSteALS/SepWater system libraries are missing. On Debian/Ubuntu run: "
                "sudo apt-get update && sudo " + " ".join(install)
                + ". Then rerun this setup cell; keep the existing output directory.")
        print("Installing missing system libraries for " + ", ".join(sorted(selected))
              + ": " + ", ".join(missing), flush=True)
        subprocess.run(["apt-get", "-qq", "update"], check=True, timeout=600)
        subprocess.run(install, check=True, timeout=600)
        found = inspect_libraries()
        missing = [package for package in requirements if package not in found]
        if missing:
            raise RuntimeError("System-library preflight failed after installation: "
                               + ", ".join(missing) + ". No model has been started.")
    print("RoSteALS/SepWater system-library preflight passed:", found, flush=True)
    return found

AUTHOR_SYSTEM_LIBRARIES = ensure_author_system_dependencies(MODELS, IN_COLAB)


In [ ]:
subprocess.run([str(CONTROLLER), "-m", "wmbench_fast.runner", "validate", "--settings", str(CONFIG)], env=ENV, check=True)


## Recover the four StegaStamp inputs

Original checkpoint, locked dependencies and recorded payloads are used. Candidate PNGs and receipts are retained in a separate Drive recovery folder. A mismatch stops without replacing inputs. This step does not rebuild digital results.


In [ ]:
# One-time repair. A repeat run checks the four original hashes and skips.
# Only known zero-byte StegaStamp files can be restored. Other inputs/results stay unchanged.
subprocess.run([str(CONTROLLER), "-u", "-m", "stegastamp_recovery_driver",
    "--project", str(PROJECT / "production"), "--bank", str(BANK_ROOT),
    "--work", str(BANK_ROOT.parent / "watermarking_stegastamp_recovery_20261016"),
    "--scratch", str(SCRATCH_ROOT / "stegastamp_recovery"),
    "--results", str(OUTPUT_ROOT)], env=ENV, check=True)


## Start / resume

First use indexes legacy records once, with visible progress. Later resumes reuse checksummed catalogs and size/mtime inventories. Do not edit the bank or result files externally in fast mode. `FULL_AUDIT=True` rehashes the underlying evidence and is intentionally slow. Pending inputs and attack ZIPs remain checksum-checked.

The controller is detached: a browser disconnect or stopping the log-view cell does not intentionally kill it. Colab VM termination still stops work; rerun with the same paths after reconnect. This cell returns immediately while computation continues.


In [ ]:
# Stop the OLD notebook runner first. Repeating this cell will not start a duplicate fast runner.
subprocess.run([str(CONTROLLER), "-m", "wmbench_fast.launch", "start", "--settings", str(CONFIG)],
               env=ENV, check=True)


## Progress

Refresh this cell manually. Per-worker telemetry is under each method's `logs/`. Reports are generated after a method finishes. Historical canary tables are not completion evidence for 100 images.


In [ ]:
# Read-only, bounded log view. Re-run whenever you want; this does not rebuild reports.
import time
from IPython.display import Markdown, display
launch = json.loads((OUTPUT_ROOT / "_reports/fast_launcher.json").read_text())
log = Path(launch["log"])
if log.exists():
    with log.open("rb") as handle:
        handle.seek(max(0, log.stat().st_size - 16000))
        print(handle.read().decode("utf-8", errors="replace"))
report = OUTPUT_ROOT / "_reports/FAST_REPORT.md"
if report.exists():
    display(Markdown(report.read_text(encoding="utf-8")))
else:
    print("Controller starting; rerun this cell shortly.")
exit_file = Path(launch["config"]).with_name(Path(launch["config"]).stem + "_exit.json")
print("Controller exit:", exit_file.read_text() if exit_file.exists() else "not yet recorded")
print("Worker logs:", OUTPUT_ROOT / "<method>/logs")


## Optional stop

Completed checkpoints remain on Drive; an interrupted case resumes from its last checkpoint. The embedding bank is read-only.


In [ ]:
# OPTIONAL. Set True only when you explicitly want to stop the background calculation.
STOP_BACKGROUND = False
if STOP_BACKGROUND:
    subprocess.run([str(CONTROLLER), "-m", "wmbench_fast.launch", "stop", "--settings", str(CONFIG)],
                   env=ENV, check=True)


## Tuning and integrity

Default: two CPU package workers and up to two GPU decoders for one method. Decoder admission waits for warm-up and memory headroom. Missing GPU telemetry means sequential admission. On A100 80 GB, test `DECODE_WORKERS=4` after reviewing two-worker throughput. Memory headroom is a heuristic, not an OOM guarantee. Do not compare concurrent decode latency with old isolated latency.

The serial reference and additional workers must agree on four fixed probes; the original per-case Clean gate remains. No valid poor result is retried. Package deletion still requires original raw-result verification. Completed methods get fast receipts so their reports and environments are not rebuilt on every resume. See `PERFORMANCE.md` inside the source ZIP for the full policy and limitations.
